# 实验4.2 深度学习网络量化实验
> **运行环境**：cann_9.0.0-py3.11-A2-arm-20260715 | ASCEND, 1*NPU 910B3, 16vCPUs, 32GiB

> **实验名称**：实验4-2 深度学习网络量化实验
> **实验环境**：GitCode 昇腾 910B Notebook（aarch64 鲲鹏 CPU + Ascend 910B NPU，PyTorch + torch_npu）
> **建议学时**：4 学时

## 实验导学

本 Notebook 将三个独立的量化示例脚本整合为一套**“边学习、边运行”**的完整实验：

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">任务</th>
<th style="text-align: left;">对应原始脚本</th>
<th style="text-align: left;">量化方式</th>
<th style="text-align: left;">核心API</th>
</tr>
<tr>
<td style="text-align: left;">任务一</td>
<td style="text-align: left;"><code>ptq_resnet18_dynamic_demo.py</code></td>
<td style="text-align: left;">训练后动态量化（Dynamic PTQ）</td>
<td style="text-align: left;"><code>quantize_dynamic</code></td>
</tr>
<tr>
<td style="text-align: left;">任务二</td>
<td style="text-align: left;"><code>ptq_resnet18_static_demo.py</code></td>
<td style="text-align: left;">训练后静态量化（Static PTQ）</td>
<td style="text-align: left;"><code>prepare</code> → 校准 → <code>convert</code></td>
</tr>
<tr>
<td style="text-align: left;">任务三</td>
<td style="text-align: left;"><code>qat_resnet18_demo.py</code></td>
<td style="text-align: left;">量化感知训练（QAT）</td>
<td style="text-align: left;"><code>prepare_qat</code> → 训练 → <code>convert</code></td>
</tr>
</table>

**三种量化方法对比详解**：
- **动态量化（Dynamic PTQ）**：最简单的方式，一次 API 调用即可完成。权重在转换时静态量化为 INT8，激活值在每次推理时动态计算 scale。无需校准数据，但只支持 Linear/LSTM 层（不支持卷积），适合 NLP 模型快速压缩。
- **静态量化（Static PTQ）**：推理前用校准数据统计各层激活值的范围，确定所有量化参数。支持卷积等大部分层，压缩效果显著（体积约降为 1/4），但需要代表性的校准数据。
- **量化感知训练（QAT）**：训练时插入伪量化节点模拟 INT8 误差，让模型适应量化噪声。精度最高但需要训练资源和时间，适合高精度要求的部署场景。

**学习目标**：
1. 理解 INT8 量化的数学原理（scale / zero_point）与量化误差来源；
2. 掌握三种主流量化方法的流程、差异与适用场景；
3. 学会在昇腾 910B 环境中**正确选择量化后端**（本环境的关键适配点）；
4. 能够完成“加载模型 → 量化 → 验证 → 对比”的完整实验闭环。

**使用方式**：按顺序自上而下运行每个单元格（Shift+Enter）。每个任务先读 Markdown 讲解，再运行代码，最后对照“结果分析”理解输出。

## 第0章 量化基础：为什么量化、量化是什么

### 0.1 为什么需要量化

深度学习模型默认使用 FP32（32位浮点）存储权重与激活值。将模型从 FP32 压缩到 **INT8（8位整数）** 可以带来：

- **体积约缩小 4 倍**：ResNet18 从约 45MB 降至约 11MB，便于部署到存储受限的设备；
- **推理加速**：整数乘加在专用硬件（如昇腾 NPU 的 AI Core）上吞吐远高于浮点；
- **降低功耗与带宽**：数据搬运量直接减为 1/4。

### 0.2 量化的数学原理

非对称仿射量化将浮点数 $r$ 映射为整数 $q$：

$$q = \mathrm{round}\left(\frac{r}{s}\right) + z, \qquad \hat{r} = s \,(q - z)$$

- $s$（scale，缩放因子）：浮点区间与整数区间的比例；
- $z$（zero_point，零点）：浮点 0 对应的整数值，保证 0 被精确表示；
- $\hat{r}$ 是反量化还原值，$\hat{r} - r$ 即**量化误差**。

### 0.3 三种量化方法一览

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">方法</th>
<th style="text-align: left;">是否需要训练</th>
<th style="text-align: left;">权重量化时机</th>
<th style="text-align: left;">激活量化时机</th>
<th style="text-align: left;">适用场景</th>
</tr>
<tr>
<td style="text-align: left;">动态 PTQ</td>
<td style="text-align: left;">否</td>
<td style="text-align: left;">转换时静态量化</td>
<td style="text-align: left;"><strong>推理时动态计算</strong></td>
<td style="text-align: left;">Linear/LSTM 为主、快速上线</td>
</tr>
<tr>
<td style="text-align: left;">静态 PTQ</td>
<td style="text-align: left;">否（仅需校准数据）</td>
<td style="text-align: left;">转换时静态量化</td>
<td style="text-align: left;"><strong>校准时统计范围</strong></td>
<td style="text-align: left;">CNN 等全网络量化</td>
</tr>
<tr>
<td style="text-align: left;">QAT</td>
<td style="text-align: left;">是（微调训练）</td>
<td style="text-align: left;">训练中学习</td>
<td style="text-align: left;">训练中学习（伪量化）</td>
<td style="text-align: left;">精度要求高的部署</td>
</tr>
</table>

**三种方法核心差异详解**：
- **动态 PTQ** 的"动态"体现在激活值的 scale 在每次推理时根据当前输入的 min/max 实时计算。由于卷积层动态计算 scale 开销过大，PyTorch 只对 Linear/LSTM 层支持动态量化，因此对 ResNet18 等卷积网络压缩效果有限。
- **静态 PTQ** 通过校准数据预先统计所有层的激活范围，确定固定量化参数。卷积层也能全 INT8 执行，体积约降为 1/4。代价是需要代表性校准数据，校准数据分布与实际推理数据差异大会导致精度下降。
- **QAT** 在训练时插入 FakeQuant 节点模拟 INT8 舍入误差，让模型在微调中适应量化噪声。最终 INT8 模型精度通常显著优于静态 PTQ，但需要训练资源和时间。

### 0.4 量化后端（backend）：本实验的关键适配点 ⚠️

PyTorch eager mode 量化的 INT8 推理在 **CPU** 上执行，具体计算由量化后端完成：

- `fbgemm`：面向 **x86** 服务器 CPU（依赖 AVX2 指令集）；
- `qnnpack`：面向 **ARM** 架构 CPU。

**GitCode 昇腾 910B 服务器通常采用鲲鹏（aarch64）CPU，fbgemm 不可用，必须使用 qnnpack**。下面的环境自检单元会自动检测 CPU 架构并选择正确的后端——这也是本实验相对原始脚本最重要的硬件适配。

## 步骤一：环境自检与量化后端适配

运行下面的单元格，确认：
1. PyTorch / torchvision 版本；
2. CPU 架构（`x86_64` 还是 `aarch64`）；
3. 自动选择量化后端：**x86 → fbgemm，ARM（鲲鹏）→ qnnpack**；
4. （可选）检测 torch_npu 与 NPU 是否可用。

> **说明**：eager mode 量化的 INT8 推理运行在 CPU 上，本实验聚焦量化方法本身；NPU 侧的量化部署（AMCT 工具 + ATC 转换）见文末拓展阅读。

In [ ]:
import platform
import warnings
import torch
import torchvision

warnings.filterwarnings("ignore")  # 屏蔽版本弃用提示，保持输出整洁

# ===== 关键顺序：先导入 torch_npu（若存在），再设置量化后端 =====
# 原因：torch_npu 导入时可能改写 torch.backends.quantized.engine，
#       若先设后导，引擎会被覆盖，导致后续 quantized::conv2d 找不到 CPU 内核。
try:
    import torch_npu
    HAS_NPU = True
except ImportError:
    HAS_NPU = False

print("PyTorch     :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("CPU 架构    :", platform.machine())
print("支持的量化引擎:", torch.backends.quantized.supported_engines)
if HAS_NPU:
    print("torch_npu   :", torch_npu.__version__, "| NPU 可用:", torch.npu.is_available())
else:
    print("torch_npu   : 未安装（量化实验在 CPU 侧进行，不影响本实验）")

# ===== 根据 CPU 架构自动选择量化后端（须在 torch_npu 导入之后）=====
# GitCode 昇腾 910B 环境多为鲲鹏 aarch64 CPU，fbgemm（x86 专用）不可用，须用 qnnpack
def setup_quant_engine():
    machine = platform.machine().lower()
    supported = torch.backends.quantized.supported_engines
    if machine in ("x86_64", "amd64") and "fbgemm" in supported:
        engine = "fbgemm"      # x86 服务器
    elif "qnnpack" in supported:
        engine = "qnnpack"     # ARM / 鲲鹏（昇腾910B宿主CPU）
    else:
        engine = supported[0]
    torch.backends.quantized.engine = engine
    return engine

ENGINE = setup_quant_engine()
print("已选择量化后端:", torch.backends.quantized.engine)

## 步骤一·五：量化算子自检（最小静态量化示例）

在跑完整 ResNet18 之前，先用一个**只有 1 个卷积层**的玩具网络走一遍静态 PTQ 全流程。
这样既能在**早期暴露**环境问题（如 `quantized::conv2d` 后端缺失），又用最少的代码展示
静态量化的五步模式：

> **包裹量化桩 → 设 qconfig → prepare → 校准 → convert**

如果这一步报错，说明环境的量化后端未配好，请回到步骤一检查 `ENGINE`；若通过，后续 ResNet18 的静态量化才有保障。

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class _SmokeNet(nn.Module):
    """仅含 1 个 Conv2d 的玩具网络，用于自检 INT8 量化是否可用"""
    def __init__(self):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()
        self.conv = nn.Conv2d(3, 8, 3, padding=1)
    def forward(self, x):
        return self.dequant(self.conv(self.quant(x)))

smoke = _SmokeNet().eval()
smoke.qconfig = torch.quantization.get_default_qconfig(ENGINE)
smoke_p = torch.quantization.prepare(smoke, inplace=False)
with torch.no_grad():
    smoke_p(torch.rand(4, 3, 8, 8))          # 校准：让 Observer 统计激活范围
smoke_q = torch.quantization.convert(smoke_p, inplace=False)

x_s = torch.rand(1, 3, 8, 8)
with torch.no_grad():
    out_float = smoke(x_s)
    out_int8 = smoke_q(x_s)
cos_s = F.cosine_similarity(out_float.flatten(), out_int8.flatten(), dim=0).item()
print(f"自检通过: INT8 Conv2d 可用 | 输出形状 {tuple(out_int8.shape)} | 余弦相似度 {cos_s:.4f}")
print("（若看到此行，说明环境的量化后端配置正确，后续静态PTQ/QAT 可放心运行）")

## 步骤二：准备公共工具函数

三个任务共用以下工具：

1. **`load_resnet18()`**：加载 ResNet18。**适配点**：Notebook 环境若无法访问外网下载 ImageNet 预训练权重（约45MB），自动回退为随机初始化权重并给出提示，保证实验不中断；
2. **`model_size_mb()`**：统计模型参数字节数，用于量化前后体积对比；
3. **`compare_outputs()`**：用余弦相似度与 Top-k 命中率衡量量化前后输出的一致性。

In [ ]:
import io
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models

# FX 图模式量化所需函数（不同 PyTorch 版本位置不同，兼容导入）
from torch.ao.quantization import quantize_fx
try:
    from torch.ao.quantization import get_default_qconfig_mapping, get_default_qat_qconfig_mapping
except (ImportError, AttributeError):
    from torch.ao.quantization.quantize_fx import get_default_qconfig_mapping, get_default_qat_qconfig_mapping

def load_resnet18():
    """加载ResNet18：优先ImageNet预训练权重，离线时回退随机初始化"""
    try:
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        src = "ImageNet 预训练权重"
    except Exception as e:
        print("[回退] 预训练权重下载失败（", type(e).__name__, "），使用随机初始化权重")
        m = models.resnet18(weights=None)
        src = "随机初始化权重"
    print("模型权重来源:", src)
    return m

def model_size_mb(m):
    """统计模型state_dict的字节大小（MB）"""
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes / 1024 / 1024

def compare_outputs(out_fp32, out_int8, topk=5):
    """对比量化前后输出：余弦相似度 + Top-k命中率"""
    cos = F.cosine_similarity(out_fp32.flatten(), out_int8.flatten(), dim=0).item()
    top_fp32 = out_fp32.topk(topk, dim=1).indices
    top_int8 = out_int8.topk(topk, dim=1).indices
    match = (top_fp32 == top_int8).float().mean().item()
    return cos, match


class QuantWrapper(nn.Module):
    """为模型添加量化输入桩(QuantStub)与输出桩(DeQuantStub)——eager 模式专用。

    步骤一·五的自检示例用此类展示了量化桩的原理。
    对于 ResNet18（带残差连接），本 Notebook 改用 FX 图模式（prepare_fx/convert_fx），
    FX 自动插桩、融合和处理残差 add，无需手动包裹。
    """
    def __init__(self, model):
        super().__init__()
        self.quant = torch.quantization.QuantStub()
        self.dequant = torch.quantization.DeQuantStub()
        self.model = model
    def forward(self, x):
        x = self.quant(x)
        x = self.model(x)
        x = self.dequant(x)
        return x

print("工具函数就绪")

## 步骤三：动手小实验——手动量化一个张量

在调用高层 API 之前，先亲手实现一遍 0.2 节的量化公式，建立直觉：

1. 统计张量的 min / max；
2. 计算 scale 与 zero_point；
3. 量化到 INT8（值域 [-128, 127]）；
4. 反量化还原，观察量化误差。

**代码说明**：
- 用 `torch.randn(8) * 2.0` 生成 8 个服从标准正态分布的浮点数（乘以 2 放大范围）。
- 按非对称仿射量化公式计算 `scale = (r_max - r_min) / (127 - (-128))` 和 `zero_point = round(-128 - r_min / scale)`。
- 量化：`q = clamp(round(r / scale) + zero_point, -128, 127)`，将浮点数映射为 INT8 整数。
- 反量化：`r_hat = scale * (q - zero_point)`，将整数还原为浮点数。
- 量化误差：`err = |r_hat - r|`，理论上每个点的误差不超过 scale/2。

**预期结果**：
- 打印原始浮点张量 r（8 个随机值，范围约 [-4, 4]）
- 打印 scale 和 zero_point（scale 约为 0.015~0.03，zero_point 约为 -100~100）
- 打印量化结果 q（8 个 INT8 整数，范围 [-128, 127]）
- 打印反量化 r_hat（8 个浮点数，与原始 r 接近但不完全相同）
- 打印绝对误差（每个点约 0.001~0.015，最大误差约为 scale/2）
- `最大误差约为 scale 的一半——这就是均匀量化的误差上界`

**观察重点**：误差大约是多少量级？哪些位置的误差更大？

In [ ]:
torch.manual_seed(42)
r = torch.randn(8) * 2.0      # 构造一个浮点张量
print("原始浮点张量 r :", r.numpy().round(4))

# 1. 统计范围
r_min, r_max = r.min().item(), r.max().item()
q_min, q_max = -128, 127      # qint8 值域

# 2. 计算 scale 与 zero_point（非对称仿射量化）
scale = (r_max - r_min) / (q_max - q_min)
zero_point = round(q_min - r_min / scale)
print(f"\nscale = {scale:.6f}, zero_point = {zero_point}")

# 3. 量化：q = round(r / scale) + zero_point，并截断到[q_min, q_max]
q = torch.clamp(torch.round(r / scale) + zero_point, q_min, q_max).to(torch.int8)
print("量化结果 q     :", q.numpy())

# 4. 反量化：r_hat = scale * (q - zero_point)
r_hat = scale * (q.float() - zero_point)
print("反量化 r_hat   :", r_hat.numpy().round(4))

# 5. 量化误差
err = (r_hat - r).abs()
print("绝对误差       :", err.numpy().round(4))
print(f"\n最大误差 {err.max():.5f}，约为 scale 的一半（{scale/2:.5f}）——这就是均匀量化的误差上界")

> **结果分析**：均匀舍入量化的单点误差不超过 scale/2。张量分布越集中（min/max 越小），scale 越小、误差越小——这解释了为什么静态 PTQ 要用**真实校准数据**统计激活范围，以及为什么 QAT 让模型在训练中“适应”量化噪声。

## 任务一：训练后动态量化（Dynamic PTQ）

> 对应原始脚本 `ptq_resnet18_dynamic_demo.py`

### 原理讲解

动态量化是**最简单**的量化方式：一次调用 `quantize_dynamic` 即可完成。

- **权重**：在转换时就量化好（静态）；
- **激活值**：不提前统计范围，而是在**每次推理时**根据当前输入动态计算 scale——故称“动态”。

**特点与适用场景**：
- 无需校准数据、无需训练，几分钟即可上线；
- 只支持 `nn.Linear`、`nn.LSTM` 等少数层类型（卷积不支持）；
- 适合全连接层占计算大头的模型（如 BERT）；对 ResNet18 这类卷积网络，仅最后一个 fc 层被量化，收益有限——本任务用它建立量化全流程的最小闭环。

### 实验步骤
1. 加载预训练 ResNet18 并设为评估模式；
2. 调用 `quantize_dynamic` 量化所有 Linear 层为 qint8；
3. 随机输入验证输出形状；
4. 对比量化前后的**模型体积**与**输出一致性**。

**代码说明（步骤1-2）**：
- `load_resnet18()` 加载模型（优先 ImageNet 预训练权重，离线时回退随机初始化）。
- `quantize_dynamic(model, {nn.Linear}, dtype=torch.qint8)` 将所有 Linear 层的权重量化为 INT8。注意只指定了 `{nn.Linear}`，卷积层不会被量化。

**预期结果（步骤1-2）**：
- `量化后 fc 层类型: DynamicQuantizedLinearPackedParams` 或类似（fc 层已变为动态量化层）
- `量化后 conv1 类型: Conv2d`（卷积层保持不变，仍为 FP32）

> **为什么 conv1 类型不变？** 因为动态量化只支持 `{nn.Linear}` 指定的层类型。ResNet18 的计算量大头是卷积层，但动态量化无法量化卷积，所以压缩效果有限。要让卷积也量化，需要任务二的静态 PTQ。

In [ ]:
# ===== 步骤1：加载预训练模型并设置评估模式 =====
model_fp32 = load_resnet18().eval()

# ===== 步骤2：动态量化（仅量化 Linear 层）=====
# 适配点：原始脚本硬编码 fbgemm，这里不显式指定后端，
# quantize_dynamic 会自动使用步骤一中按架构选择好的 ENGINE
model_dyn = torch.quantization.quantize_dynamic(
    model_fp32,
    {torch.nn.Linear},   # 指定要量化的层类型
    dtype=torch.qint8,   # 权重量化为8位整数
)

# 观察：fc 层已变为 DynamicQuantizedLinear，卷积层保持浮点
print("量化后 fc 层类型:", type(model_dyn.fc).__name__)
print("量化后 conv1 类型:", type(model_dyn.conv1).__name__)

In [ ]:
# ===== 步骤3：推理验证 =====
x = torch.rand(1, 3, 224, 224)
with torch.no_grad():
    out_fp32 = model_fp32(x)
    out_dyn = model_dyn(x)
print("FP32 输出形状  :", out_fp32.shape)
print("INT8 输出形状  :", out_dyn.shape)

# ===== 步骤4：体积与一致性对比 =====
cos, match = compare_outputs(out_fp32, out_dyn, topk=5)
print(f"\n输出余弦相似度: {cos:.4f}（越接近1越一致）")
print(f"Top-5 命中率  : {match:.2f}")
print(f"模型体积: FP32 = {model_size_mb(model_fp32):.1f} MB -> 动态INT8 = {model_size_mb(model_dyn):.1f} MB")

> **结果分析**：输出形状 `[1, 1000]` 与 ImageNet 的 1000 个类别对应。ResNet18 只有一个 fc 层（约 0.5M 参数），所以体积只下降了约 3%——这正是动态量化"只量化 Linear"的局限。想让整个网络（尤其是占大头的卷积层）都量化，需要任务二的静态 PTQ。

**预期输出数值说明**：
- `FP32 输出形状: torch.Size([1, 1000])`
- `INT8 输出形状: torch.Size([1, 1000])` —— 形状不变，量化不改变输出维度
- `输出余弦相似度: 约 0.99~1.00`（fc 层量化对整体输出影响很小）
- `Top-5 命中率: 约 0.80~1.00`（量化前后 Top-5 预测类别基本一致）
- `模型体积: FP32 ≈ 43 MB -> 动态INT8 ≈ 43 MB`（仅 fc 层量化，体积几乎不变）

> **为什么体积几乎不变？** ResNet18 的参数大部分在卷积层（约 11M 参数），fc 层只有 512×1000=512K 参数。动态量化只量化了 fc 层的权重（512K×4B→512K×1B，节省约 1.5MB），相对于 43MB 的总体积，节省比例仅约 3%。

## 任务二：训练后静态量化（Static PTQ）

> 对应原始脚本 `ptq_resnet18_static_demo.py`

### 原理讲解

静态 PTQ 在推理**之前**就确定所有量化参数，推理全程使用 INT8 计算，流程为三步：

1. **准备（prepare）**：按 `qconfig` 在网络的输入输出处插入 **Observer（观测器）**，它不量化，只统计流经张量的 min/max；
2. **校准（calibration）**：用一批**有代表性的数据**前向跑若干轮，Observer 收集激活值的分布范围，据此计算每层的 scale 与 zero_point；
3. **转换（convert）**：把浮点模块替换为量化模块（如 `QuantizedConv2d`），权重永久转为 INT8。

**与动态量化的关键区别**：激活的量化参数来自**校准统计**而非运行时动态计算，因此卷积等算子也能全 INT8 执行，压缩与加速收益大得多——代价是需要校准数据，且校准数据的质量直接影响精度。

### 为什么用 FX 图模式量化（而非 eager 模式）

步骤一·五的自检示例用 **eager 模式**（手动插 QuantStub/DeQuantStub）展示了量化桩的原理。
对于 ResNet18 这类带**残差连接**（`out += identity`）的模型，eager 模式会遇到困难：
`convert` 后残差加法的两个操作数都是量化张量（`QuantizedCPU` 后端），而 `aten::add`
没有 `QuantizedCPU` 内核 -> 报 `NotImplementedError`。

**FX 图模式**（`prepare_fx` / `convert_fx`）通过符号追踪获取完整计算图，能自动：
1. 在合适位置插入 `QuantStub` / `DeQuantStub`（无需手动包裹）；
2. 融合 `Conv+BN+ReLU` 为单一算子；
3. 在残差 `add` 前后自动插入 `dequant → add → quant`，避免量化张量直接相加。

> 简单模型（无残差连接）可用 eager 模式手动插桩；复杂模型推荐 FX 图模式。

### 实验步骤
1. 加载模型，用 `prepare_fx` 追踪计算图并插入 Observer（自动融合 + 自动插桩）；
2. 用随机数据做 10 轮校准（实际应用应使用真实数据）；
3. `convert_fx` 得到 INT8 模型，验证输出并对比体积。

In [ ]:
# ===== 步骤1：加载模型 + FX 图模式量化配置 =====
model_fp32 = load_resnet18().eval()

# FX 图模式（prepare_fx）自动完成：① 符号追踪计算图 ② 插入 QuantStub/DeQuantStub
# ③ 融合 Conv+BN+ReLU ④ 在残差 add 前后插入 dequant/quant
example_inputs = (torch.rand(1, 3, 224, 224),)
qconfig_mapping = get_default_qconfig_mapping(ENGINE)
model_prepared = quantize_fx.prepare_fx(model_fp32, qconfig_mapping, example_inputs)
print("prepare_fx 完成：Observer 已插入，Conv+BN+ReLU 已自动融合")

In [ ]:
# ===== 步骤3：校准（calibration）=====
# 用10批随机数据前向传播，Observer 统计各层激活的 min/max
# 注意：实际工程中必须改用真实训练/验证数据，随机数据仅为流程演示
with torch.no_grad():
    for i in range(10):
        inputs = torch.rand(32, 3, 224, 224)
        model_prepared(inputs)
        print(f"校准批次 {i+1}/10 完成", end="\r")
print("\n校准完成")

In [ ]:
# ===== 观察：查看 Observer 统计到的激活范围 =====
# FX 模式下 Observer 可能是独立模块（有 min_val），也可能挂在 .activation_post_process 上
# 收集所有 Observer，优先找 fc 相关的，找不到则取最后一个（最接近输出）
observers = []
for name, module in model_prepared.named_modules():
    if hasattr(module, "min_val") and hasattr(module, "max_val"):
        observers.append((name, module))
    ap = getattr(module, "activation_post_process", None)
    if ap is not None and hasattr(ap, "min_val"):
        observers.append((name + ".activation_post_process", ap))

obs, obs_name = None, "?"
for name, o in observers:
    if "fc" in name:
        obs, obs_name = o, name
        break
if obs is None and observers:
    obs_name, obs = observers[-1]

print(f"Observer [{obs_name}] 激活范围: min = {obs.min_val.item():.4f}, max = {obs.max_val.item():.4f}")
scale, zp = obs.calculate_qparams()
print(f"据此计算的量化参数: scale = {scale.item():.6f}, zero_point = {zp.item()}")

In [ ]:
# ===== 步骤4：convert_fx —— 转换为真正的 INT8 模型 =====
model_static = quantize_fx.convert_fx(model_prepared)
print("转换后 conv1 类型:", type(model_static.conv1).__name__, "（卷积也已量化！）")
print("转换后 fc 类型   : ", type(model_static.fc).__name__)

# ===== 推理验证与对比 =====
x = torch.rand(1, 3, 224, 224)
with torch.no_grad():
    out_fp32 = model_fp32(x)
    out_static = model_static(x)
print("\nPTQ INT8 输出形状:", out_static.shape)

cos, match = compare_outputs(out_fp32, out_static, topk=5)
print(f"输出余弦相似度: {cos:.4f} | Top-5 命中率: {match:.2f}")
print(f"模型体积: FP32 = {model_size_mb(model_fp32):.1f} MB -> 静态INT8 = {model_size_mb(model_static):.1f} MB")

> **结果分析**：静态 INT8 模型体积约为 FP32 的 1/4（约 45MB → 约 11MB），因为**全部卷积层**都被量化了——这是静态 PTQ 相比动态量化的核心优势。
>
> **预期输出数值说明**：
> - `转换后 conv1 类型: QuantizedConv2d`（卷积层已量化为 INT8！）
> - `转换后 fc 类型: QuantizedLinear`（全连接层也已量化）
> - `PTQ INT8 输出形状: torch.Size([1, 1000])`
> - `输出余弦相似度: 约 0.95~1.00`（全网络量化后相似度略低于动态量化，但仍在可接受范围）
> - `Top-5 命中率: 约 0.60~1.00`（随机校准数据可能导致精度波动）
> - `模型体积: FP32 ≈ 43 MB -> 静态INT8 ≈ 11 MB`（约 1/4，因为所有卷积和全连接层权重都从 FP32 变为 INT8）
>
> **为什么体积降为 1/4？** FP32 每个权重占 4 字节，INT8 占 1 字节，4/1=4。静态 PTQ 将所有卷积层和全连接层的权重都量化为 INT8，因此体积约为原来的 1/4。注意 bias 等少量参数仍为 FP32，所以实际略大于严格 1/4。
>
> **进阶（可选）**：`prepare_fx` 在追踪计算图时已**自动融合** Conv+BN+ReLU 为单一算子，无需手动调用 `fuse_fx`。下面的单元格观察已转换模型中实际出现的量化/融合算子类型。

In [ ]:
# ===== 进阶说明：prepare_fx 已自动融合 Conv+BN+ReLU =====
# 上面的 prepare_fx 在追踪计算图时已自动将 Conv+BN+ReLU 融合为单一算子，
# 无需手动调用 fuse_fx。下面观察已转换模型中的算子类型分布：
from collections import Counter
type_counter = Counter()
for name, module in model_static.named_modules():
    if module is model_static or len(list(module.children())) > 0:
        continue  # 跳过顶层和容器模块
    mod_ns = type(module).__module__
    cls_name = type(module).__name__
    label = f"[INT8] {cls_name}" if "quantized" in mod_ns else f"[FP32] {cls_name}"
    type_counter[label] += 1

print("静态PTQ 模型中的算子类型分布：")
for label, count in sorted(type_counter.items()):
    print(f"  {label}: {count} 个")
print(f"\n模型体积: {model_size_mb(model_static):.1f} MB（约为 FP32 的 1/4）")

## 任务三：量化感知训练（QAT）

> 对应原始脚本 `qat_resnet18_demo.py`

### 原理讲解

静态 PTQ 的量化参数在训练后“一次性”确定，模型从未“见过”量化噪声，精度损失不可控。QAT（Quantization-Aware Training）的思路是：

- 训练时在 forward 中插入 **FakeQuantize（伪量化）** 节点：数据先“量化→立即反量化”，**模拟** INT8 的舍入误差，但计算仍用浮点进行，因此可以正常反向传播；
- 模型在微调中逐渐**适应**量化噪声，学出对量化更鲁棒的权重；
- 训练结束后 `convert` 为真 INT8 模型，精度通常显著优于静态 PTQ。

**三步流程**：`prepare_qat`（训练态插入伪量化）→ 微调训练若干轮 → `convert`（评估态转 INT8）。

### 实验步骤
1. 加载模型，用 `prepare_qat_fx` 追踪计算图并插入伪量化节点（自动融合 + 自动插桩）；
2. 用随机数据微调 2 轮（演示流程；实际应用使用真实数据）；
3. 切到 eval 模式后 `convert_fx`，验证 INT8 推理。

In [ ]:
import torch.optim as optim
# ===== 步骤1：加载预训练模型作为 QAT 起点 =====
model_fp32 = load_resnet18()

# ===== 步骤2：FX 图模式 QAT =====
# prepare_qat_fx 自动：符号追踪 + 融合 Conv+BN+ReLU + 插入 FakeQuantize + 处理残差连接
example_inputs = (torch.rand(1, 3, 224, 224),)
qat_qconfig_mapping = get_default_qat_qconfig_mapping(ENGINE)
model_qat = quantize_fx.prepare_qat_fx(model_fp32.train(), qat_qconfig_mapping, example_inputs)

# 观察：conv1 已带 FakeQuantize
print("QAT conv1 类型:", type(model_qat.conv1).__name__)
for n, m in model_qat.named_modules():
    if hasattr(m, "weight_fake_quant"):
        print(f"伪量化示例: {n} -> {type(m).__name__} | 权重伪量化器: {type(m.weight_fake_quant).__name__}")
        break

In [ ]:
# ===== 步骤3：量化感知训练（2轮，演示流程）=====
optimizer = optim.SGD(model_qat.parameters(), lr=1e-3, momentum=0.9)
criterion = nn.CrossEntropyLoss()
losses = []

for epoch in range(2):
    # 生成示例训练数据（实际应用请替换为真实数据集）
    inputs = torch.rand(32, 3, 224, 224)
    targets = torch.randint(0, 1000, (32,))
    # 前向传播：包含伪量化，模拟 INT8 噪声
    outputs = model_qat(inputs)
    loss = criterion(outputs, targets)
    # 反向传播与参数更新（浮点进行）
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    print(f"Epoch {epoch}, loss = {loss.item():.4f}")

In [ ]:
# ===== 步骤4：转换 + 验证 =====
# 注意：convert 前必须先 eval()，让 BN 使用统计量、伪量化参数冻结
model_int8 = quantize_fx.convert_fx(model_qat.eval())

x = torch.rand(1, 3, 224, 224)
with torch.no_grad():
    out_qat = model_int8(x)
print("QAT INT8 输出形状:", out_qat.shape)
print(f"QAT 模型体积: {model_size_mb(model_int8):.1f} MB")
print(f"训练损失变化: {losses[0]:.4f} -> {losses[-1]:.4f}（随机标签下仅供参考流程）")

## 总结：三种量化方法对比

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">维度</th>
<th style="text-align: left;">动态 PTQ（任务一）</th>
<th style="text-align: left;">静态 PTQ（任务二）</th>
<th style="text-align: left;">QAT（任务三）</th>
</tr>
<tr>
<td style="text-align: left;">核心 API</td>
<td style="text-align: left;"><code>quantize_dynamic</code></td>
<td style="text-align: left;"><code>prepare</code>+校准+<code>convert</code></td>
<td style="text-align: left;"><code>prepare_qat</code>+训练+<code>convert</code></td>
</tr>
<tr>
<td style="text-align: left;">需要数据</td>
<td style="text-align: left;">不需要</td>
<td style="text-align: left;">需要校准数据（不训练）</td>
<td style="text-align: left;">需要训练数据（微调）</td>
</tr>
<tr>
<td style="text-align: left;">量化层类型</td>
<td style="text-align: left;">仅 Linear/LSTM</td>
<td style="text-align: left;">Conv/Linear 等大部分层</td>
<td style="text-align: left;">同静态 PTQ</td>
</tr>
<tr>
<td style="text-align: left;">激活量化时机</td>
<td style="text-align: left;">推理时动态计算</td>
<td style="text-align: left;">校准时统计确定</td>
<td style="text-align: left;">训练中学习确定</td>
</tr>
<tr>
<td style="text-align: left;">精度保持</td>
<td style="text-align: left;">较好（Linear影响小）</td>
<td style="text-align: left;">依赖校准数据质量</td>
<td style="text-align: left;"><strong>最好</strong></td>
</tr>
<tr>
<td style="text-align: left;">实施成本</td>
<td style="text-align: left;">极低</td>
<td style="text-align: left;">低</td>
<td style="text-align: left;">较高（需训练资源）</td>
</tr>
<tr>
<td style="text-align: left;">ResNet18 体积</td>
<td style="text-align: left;">≈43MB（仅fc量化）</td>
<td style="text-align: left;">≈11MB（全网络量化）</td>
<td style="text-align: left;">≈11MB</td>
</tr>
<tr>
<td style="text-align: left;">典型场景</td>
<td style="text-align: left;">NLP/全连接模型快速部署</td>
<td style="text-align: left;">CNN 推理部署</td>
<td style="text-align: left;">高精度要求的端边部署</td>
</tr>
</table>

**对比表详解**：
- **精度保持**：动态 PTQ 只量化 Linear，对 CNN 影响小但压缩有限；静态 PTQ 全网络量化，精度依赖校准数据质量；QAT 在训练中适应量化噪声，精度最好。
- **实施成本**：动态 PTQ 一行代码即可；静态 PTQ 需准备校准数据走 prepare→calibrate→convert 三步；QAT 还需额外训练资源和微调时间。
- **ResNet18 体积**：动态 PTQ 约 43MB（仅 fc 量化，几乎不变）；静态 PTQ 和 QAT 都约 11MB（全网络量化，降为 1/4）。
- **选择建议**：NLP 模型用动态 PTQ 快速上线；CNN 推理用静态 PTQ 性价比最高；精度要求严格时用 QAT。

**实验结论**：
1. 三种方法殊途同归——最终都产出 INT8 模型并用 `shape=[1,1000]` 的推理验证；
2. 动态 PTQ 最简单但覆盖面窄；静态 PTQ 性价比最高；QAT 精度最好但成本最高；
3. 在昇腾 910B（鲲鹏 ARM）环境，量化后端必须选 `qnnpack`；eager mode 量化推理在 CPU 进行，NPU 侧部署需走 AMCT/ATC 工具链。

## 思考题

1. 动态量化为什么不支持卷积层？从“激活 scale 动态计算”的开销角度思考。
2. 静态 PTQ 校准数据从 10 批减到 1 批，精度和 Observer 统计会有什么变化？动手试一试。
3. 为什么 QAT 的 `convert` 之前必须先调用 `eval()`？不调用会发生什么？
4. 本实验用随机数据校准/训练，若换成真实 ImageNet 数据，三种方法的 Top-5 一致性会有什么变化？为什么？
5. 若要把 INT8 模型部署到昇腾 910B NPU 上，还需要哪些工具链环节？（提示：AMCT、ATC、离线模型 om）

## 常见问题

<table style="text-align: left; margin-left: 0;">
<tr>
<th style="text-align: left;">问题</th>
<th style="text-align: left;">解决办法</th>
</tr>
<tr>
<td style="text-align: left;"><code>NotImplementedError: Could not run 'quantized::conv2d.new'</code></td>
<td style="text-align: left;">eager 模式缺少量化桩或后端被 <code>torch_npu</code> 覆盖。本 Notebook 已改用 FX 图模式（自动插桩）并在 <code>torch_npu</code> 导入后设置 <code>engine</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>NotImplementedError: Could not run 'aten::add.out' ... QuantizedCPU</code></td>
<td style="text-align: left;">ResNet 残差连接中两个量化张量直接相加。改用 FX 图模式（<code>prepare_fx</code>/<code>convert_fx</code>），FX 自动在 add 前后插入 dequant/quant</td>
</tr>
<tr>
<td style="text-align: left;">预训练权重下载失败</td>
<td style="text-align: left;">环境无外网，工具函数已自动回退随机权重；或提前离线下载 <code>resnet18-f37072fd.pth</code> 放入 <code>~/.cache/torch/hub/checkpoints/</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>pretrained=True</code> 报 DeprecationWarning</td>
<td style="text-align: left;">新版 torchvision 写法：<code>weights=models.ResNet18_Weights.IMAGENET1K_V1</code>，本 Notebook 已采用</td>
</tr>
<tr>
<td style="text-align: left;">校准内存不足</td>
<td style="text-align: left;">将校准 batch 从 32 降到 8，或减少校准批数</td>
</tr>
<tr>
<td style="text-align: left;">量化后精度掉得厉害</td>
<td style="text-align: left;">检查校准数据是否为真实分布；尝试模块融合；改用 QAT</td>
</tr>
</table>

## 拓展阅读（昇腾 NPU 部署路径）

- **AMCT**（Ascend Model Compression Toolkit）：昇腾官方模型压缩工具，支持训练后量化与量化感知训练，产出适配 NPU 的量化模型；
- **ATC**（Ascend Tensor Compiler）：将模型编译为昇腾离线模型 `.om`，在 910B 上经 CANN 运行时执行 INT8 推理；
- 社区入口：CANN 社区（hiascend.com）、昇腾样例仓（gitee.com/ascend/samples）。

---

## 课后练习

请根据本实验内容完成以下题目进行自测。


**第1题**（单选题）INT8 量化的数学公式中，scale 的含义是？

- A. 零点
- B. 缩放因子，浮点区间与整数区间的比例
- C. 量化误差
- D. 数据范围


In [ ]:
q1 = ''  # 填入你的选项，如 'B'
print(f'第1题答案已记录：{q1}' if q1 else '请填入答案并运行本单元格')

**第2题**（单选题）动态量化的特点是？

- A. 需要校准数据
- B. 激活值在推理时动态计算 scale
- C. 需要训练
- D. 只支持卷积层


In [ ]:
q2 = ''  # 填入你的选项，如 'B'
print(f'第2题答案已记录：{q2}' if q2 else '请填入答案并运行本单元格')

**第3题**（单选题）静态 PTQ 的三个步骤是？

- A. 训练→量化→验证
- B. prepare → 校准 → convert
- C. 量化→训练→转换
- D. prepare → 训练 → convert


In [ ]:
q3 = ''  # 填入你的选项，如 'C'
print(f'第3题答案已记录：{q3}' if q3 else '请填入答案并运行本单元格')

**第4题**（单选题）QAT（量化感知训练）中插入的节点是？

- A. Observer
- B. FakeQuantize（伪量化）
- C. QuantStub
- D. DeQuantStub


In [ ]:
q4 = ''  # 填入你的选项，如 'B'
print(f'第4题答案已记录：{q4}' if q4 else '请填入答案并运行本单元格')

**第5题**（单选题）在鲲鹏 ARM 架构上，应选择哪个量化后端？

- A. fbgemm
- B. qnnpack
- C. none
- D. default


In [ ]:
q5 = ''  # 填入你的选项，如 'B'
print(f'第5题答案已记录：{q5}' if q5 else '请填入答案并运行本单元格')

**第6题**（单选题）动态量化只支持哪些层类型？

- A. Conv2d 和 Linear
- B. Linear 和 LSTM
- C. 所有层
- D. 只有 Conv2d


In [ ]:
q6 = ''  # 填入你的选项，如 'B'
print(f'第6题答案已记录：{q6}' if q6 else '请填入答案并运行本单元格')

**第7题**（单选题）静态 PTQ 相比动态量化的优势是？

- A. 更简单
- B. 卷积层也能全 INT8 执行，压缩更大
- C. 不需要数据
- D. 精度更高


In [ ]:
q7 = ''  # 填入你的选项，如 'B'
print(f'第7题答案已记录：{q7}' if q7 else '请填入答案并运行本单元格')

**第8题**（单选题）QAT 的 convert 之前必须先调用什么？

- A. train()
- B. eval()
- C. cpu()
- D. cuda()


In [ ]:
q8 = ''  # 填入你的选项，如 'B'
print(f'第8题答案已记录：{q8}' if q8 else '请填入答案并运行本单元格')

**第9题**（单选题）将 ResNet18 从 FP32 量化到 INT8，体积大约变为原来的？

- A. 1/2
- B. 1/4
- C. 1/8
- D. 不变


In [ ]:
q9 = ''  # 填入你的选项，如 'B'
print(f'第9题答案已记录：{q9}' if q9 else '请填入答案并运行本单元格')

**第10题**（单选题）FX 图模式量化的优势是？

- A. 更简单
- B. 自动插桩、融合算子、处理残差连接
- C. 不需要数据
- D. 精度更高


In [ ]:
q10 = ''  # 填入你的选项，如 'B'
print(f'第10题答案已记录：{q10}' if q10 else '请填入答案并运行本单元格')

**全部作答完成后，运行下方代码查看批改结果：**


In [ ]:
import sys
from pathlib import Path

for candidate in (Path.cwd() / 'answer', Path.cwd().parent / 'answer'):
    if candidate.exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError('Cannot find answer directory')
from grade_04 import grade
grade(globals())

## 参考资料

- [昇腾社区 - AMCT 模型压缩](https://hiascend.com/document)
- [PyTorch 量化教程](https://pytorch.org/tutorials/)
